In [1]:
#%% import libraries

import os
import sys
import time
from time import sleep
import  threading
from multiprocessing import Queue

from IPython.display import clear_output

from pypylon import pylon
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import datetime


In [2]:

# %% def the function make_acquisition (for a given frequency, for all chosen cameras)
def make_acquisition(freq_acq, cam_array, list_camera_SN, acquisition_folder, save_images=True,numberOfImagesToGrab=100):
    
    try:
        cam_array.Open()

        # demonstrate some feature access
        list_str_camera_serial_number = []
        #list_freq_acq_effective = []
        for idx_cam in range(len(list(cam_array))):
            camera = cam_array[idx_cam]
            print(camera.Width.GetInc())
            print(camera.Width.GetValue())

            camera.AcquisitionFrameRate.SetValue(freq_acq)  # Replace with your desired frame rate
            camera.AcquisitionFrameRateEnable.SetValue(True)            
            print('frame rate : ',camera.ResultingFrameRate.GetValue())

            # mettre un buffer assez grand :
            camera.MaxNumBuffer.SetValue(9000)
            str_camera_serial_number = camera.DeviceSerialNumber()
            list_str_camera_serial_number.append(str_camera_serial_number)
            #list_freq_acq_effective.append(freq_acq_effective)
            savefolder = f'{acquisition_folder}/camera_{str_camera_serial_number}/{str(np.round(freq_acq,4))}Hz/'

            ## create directory to save images
            if save_images:
                if os.path.exists(acquisition_folder)==False:
                    os.mkdir(acquisition_folder)
                if os.path.exists(acquisition_folder+'camera_'+str_camera_serial_number+'/')==False:
                    os.mkdir(acquisition_folder+'camera_'+str_camera_serial_number+'/')
                if os.path.exists(savefolder)==False:
                    os.mkdir(savefolder)
                if os.path.exists(savefolder + 'images/')==False:
                    os.mkdir(savefolder + 'images/')
            ## write camera settings in a txt file    
            file = open(savefolder + 'camera_settings.txt','w')
            file.write('Serial Number : '+ list_camera_SN[idx_cam]+' \n')
            file.write('Exposure Time : ' + str(camera.ExposureTime.GetValue())+' microseconds \n')
            file.write('Width : ' + str(camera.Width.GetValue())+' \n')  # Replace with your desired width
            file.write('Height : '+str(camera.Height.GetValue())+' \n')  # Replace with your desired height
            file.write('OffsetX : '+str(camera.OffsetX.GetValue())+' \n')  # Replace with your desired X offset
            file.write('OffsetY : '+str(camera.OffsetY.GetValue())+' \n')  # Replace with your desired Y offset
            file.write('Max Buffer Size : '+str(camera.MaxNumBuffer.GetValue())+' \n')
            file.close()
            
        # note the date and time of the beginning of the acquisition (useful in file names)
        time_acquisition = datetime.datetime.now()
        year = time_acquisition.year
        month = time_acquisition.month
        day = time_acquisition.day
        hour = time_acquisition.hour
        minutes = time_acquisition.minute
        seconds = time_acquisition.second
        str_time_acquisition = str(year)+str(month).zfill(2)+str(day).zfill(2)+'_'+str(hour).zfill(2)+str(minutes).zfill(2)+str(seconds).zfill(2)
        
        
        # start to grab images for all cameras at the same time
        #cam_array.StartGrabbing(numberOfImagesToGrab)
        array_time_elapsed = []
        time_init = time.time()
        for camera in cam_array:
            camera.StartGrabbing()
            array_time_elapsed.append(time.time() - time_init)
        array_time_elapsed = np.array(array_time_elapsed)
        print(array_time_elapsed)
        array_time_elapsed_wrtcam0 = array_time_elapsed - array_time_elapsed[0]
        array_num_images_elapsed = array_time_elapsed_wrtcam0//(1/freq_acq)
        print(array_time_elapsed_wrtcam0)
        print(array_num_images_elapsed)
            
        i = 0

        while cam_array[0].IsGrabbing(): # on a dû adapter :/ (normalement c'était juste cam_array.IsGrabbing)
            for idx_cam in range(len(list(cam_array))):
                camera = cam_array[idx_cam]
                grabResult = camera.RetrieveResult(5000, pylon.TimeoutHandling_ThrowException)
                if grabResult.GrabSucceeded():
                    # Access the image data.
                    print('idx_cam =',idx_cam)
                    #print("SizeX: ", grabResult.Width)
                    #print("SizeY: ", grabResult.Height)
                    img = grabResult.Array
                    #print("Gray value of first pixel: ", img[0, 0])
                    if save_images==True:
                        # Convert the NumPy array to a PIL Image object
                        image = Image.fromarray(img)
                        # Save the image as a TIFF file (black and white)
                        str_camera_serial_number = list_str_camera_serial_number[idx_cam]
                        savefolder = acquisition_folder+'camera_'+str_camera_serial_number+'/' + str(np.round(freq_acq,4))+'Hz/'

                        namefile = 'Basler__'+str_camera_serial_number+'__'+str_time_acquisition+'_'+str(int(i+array_num_images_elapsed[idx_cam]-array_num_images_elapsed[-1])).zfill(4)+'.tiff' # pour les cameras qui commencent avant les autres, les premieres images auront des nombres négatifs
                        #namefile = 'Basler__'+str_camera_serial_number+'__'+str_time_acquisition+'_'+str(int(i)).zfill(4)+'.tiff'
                        if (int(i+array_num_images_elapsed[idx_cam]-array_num_images_elapsed[-1]) < 0)|(int(i+array_num_images_elapsed[idx_cam]-array_num_images_elapsed[-1])>=numberOfImagesToGrab):
                            pass
                        else:
                            image.save(savefolder+'images/'+namefile)
                            print('image number '+str(i))
                            clear_output(wait=True)
                            
                        
            grabResult.Release()
            print(i)
            i+=1
            if i>=numberOfImagesToGrab + array_num_images_elapsed[-1]:
                cam_array.StopGrabbing()
                cam_array.Close()            

    finally:
        # Ensure camera is closed properly
        #cam_array.StopGrabbing()
        cam_array.Close()



In [3]:
######################################################
# SCAN DES DISPOSITIFS USB POUR TROUVER LA/LES CAMERAS
######################################################

# Create a CameraFactory instance
tl_factory = pylon.TlFactory.GetInstance()

# Choose the devices to use
device_count = tl_factory.EnumerateDevices()
NUM_MAX_CAMERAS = len(list(device_count))

list_camera_SN = ['40232065']

#cam_array_temp = pylon.InstantCameraArray(NUM_MAX_CAMERAS)
NUM_CAMERAS = len(list_camera_SN)
cam_array = pylon.InstantCameraArray(NUM_CAMERAS)

cam_info = tl_factory.GetInstance().EnumerateDevices()
selected_cameras_indices = []
for i in range(len(list(cam_info))):
    camera = cam_info[i]
    if camera.GetSerialNumber() in list_camera_SN:
        selected_cameras_indices.append(i)

#print(selected_cameras_indices)

j=0
for i in selected_cameras_indices:
    cam_array[j].Attach(tl_factory.CreateDevice(device_count[i]))
    cam_array[j].Open()
    print('camera with index '+str(j) + ' has seral number : '+ cam_array[j].DeviceSerialNumber())
    cam_array[j].Close()
    j+=1

if len(device_count)==0:
    print('0 camera found')



##################################################### INPUTS & SETTINGS ######################################################
freq_acq = 15

# %% choose number of images to grab and the name of the daily folder
numberOfImagesToGrab = 9000 # choisir nombre d'images
save_images = True

#############################################################################################################################




0 camera found


In [5]:
#%% ensuite assigner à chaque caméra une liste pour chaque parametre à changer (les listes doivent chacune avoir NUM_CAMERAS elements)

#camera.MaxNumBuffer.SetValue(desired_buffer_size) (fonction pour choisir buffer size)

ddd = datetime.datetime.now()
date = str(ddd.year) + str(ddd.month).zfill(2) + str(ddd.day).zfill(2)

datefolder = f'C:/Users/Vasco Zanchi/Desktop/CASCADE26/tests_interfacage_test_bresilien/{date}/'
dailyfolder = f'{datefolder}/tests_bresilens/'

if os.path.exists(datefolder)==False:
    os.mkdir(datefolder)
if os.path.exists(dailyfolder)==False:
    os.mkdir(dailyfolder)



num_acquisition = 1
acquisition_folder = dailyfolder + 'Acquisition_' + str(num_acquisition) + '/'
while ((os.path.exists(acquisition_folder))|(os.path.exists(f'{dailyfolder}/Acquisition_{str(num_acquisition)}/'))):
    num_acquisition += 1
    acquisition_folder = dailyfolder + 'Acquisition_' + str(num_acquisition) + '/'

print("Acquisition number : "+str(num_acquisition))




Acquisition number : 1


## Test d'acquisition d'images

In [6]:
make_acquisition(10, cam_array=cam_array, list_camera_SN=list_camera_SN, acquisition_folder=dailyfolder, save_images=True, numberOfImagesToGrab=10)

9


## Test lecture data PhidgetBridge

In [8]:
from icewave.vasco.field.CASCADE26.Fracture_labo import interface_phidget_bridge_forbraziliantest
%matplotlib qt
interface_phidget_bridge_4inputs.main()


Fichier existant, ajout des données...


C:\Users\Vasco Zanchi\Documents\git_turbotice\icewave\icewave\vasco\field\CASCADE26\Fracture_labo\interface_phidget_bridge_forbraziliantest.py:103: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax.set_xlim(max(0, t - remove_interval), t)


KeyboardInterrupt: 

#### marche pas pour l'instant :
### Code pour aller lire pendant que les images sont enregistrées dans un dossier, régulièrement, la dernière image enregistrée

In [8]:
import os
import time
import re
import threading
import matplotlib.pyplot as plt
from matplotlib.image import imread
from natsort import natsorted
import numpy as np

# --- Configuration ---
# Exemple : list_camera_SN = ["40232065", "40232066", ...] (à définir selon vos caméras)
list_camera_SN = ["40232065"]  # Remplacez par vos numéros de série
dossier_base = "C:/Users/Vasco Zanchi/Desktop/CASCADE26/tests_interfacage_test_bresilien/20261016/tests_bresilens/"
freq_acq = 3  # Fréquence d'acquisition (Hz)
intervalle_affichage = 0.5  # Intervalle de rafraîchissement (en secondes)
derniers_chiffres_precedent = {idx: None for idx in range(len(list_camera_SN))}  # Dictionnaire pour suivre les images

# --- Initialisation de la figure matplotlib ---
plt.ion()  # Mode interactif ON
fig, axes = plt.subplots(1, len(list_camera_SN), figsize=(12, 6))
if len(list_camera_SN) == 1:
    axes = [axes]  # Si une seule caméra, axes n'est pas une liste
plt.suptitle("Affichage en temps réel des caméras Basler")

# --- Fonction pour extraire les 4 derniers chiffres ---
def extraire_derniers_chiffres(nom_fichier):
    match = re.search(r'_\d{4}\.tiff$', nom_fichier)
    return match.group(0)[1:-5] if match else None

# --- Fonction pour afficher les images ---
def afficher_images():
    while True:
        for idx_cam, cam_SN in enumerate(list_camera_SN):
            # Chemin vers le dossier de la caméra
            dossier_camera = os.path.join(dossier_base, f"camera_{cam_SN}", f"{freq_acq}Hz", "images")
            if not os.path.exists(dossier_camera):
                print(f"Dossier introuvable : {dossier_camera}")
                continue

            # Lister les fichiers TIFF pour cette caméra
            fichiers = [f for f in os.listdir(dossier_camera) if f.startswith(f'Basler__{cam_SN}__') and f.endswith('.tiff')]
            if not fichiers:
                continue

            # Trier les fichiers par nom (ordre naturel)
            fichiers_tries = natsorted(fichiers)
            dernier_fichier = fichiers_tries[-1]
            derniers_chiffres = extraire_derniers_chiffres(dernier_fichier)

            # Vérifier si c'est une nouvelle image
            if derniers_chiffres != derniers_chiffres_precedent[idx_cam]:
                derniers_chiffres_precedent[idx_cam] = derniers_chiffres
                chemin_image = os.path.join(dossier_camera, dernier_fichier)

                try:
                    img = imread(chemin_image)
                    axes[idx_cam].clear()
                    axes[idx_cam].imshow(img, cmap='gray' if img.ndim == 2 else None)  # Gestion des images en niveaux de gris
                    axes[idx_cam].axis('off')
                    axes[idx_cam].set_title(f"Cam {cam_SN}: {dernier_fichier}")
                    fig.canvas.draw()  # Met à jour la figure
                    fig.canvas.flush_events()  # Force le rafraîchissement
                except Exception as e:
                    print(f"Erreur avec {chemin_image} : {e}")

        time.sleep(intervalle_affichage)

def afficher_images_et_dataphidget(name_phidget_file=r"C:\Users\Vasco Zanchi\Desktop\CASCADE26\tests_interfacage_test_bresilien\20261016\tests_bresilens\PhidgetBridge\test1.txt"):
    while True:
        # =====================================================
        # 1. AFFICHAGE DES IMAGES
        # =====================================================
        for idx_cam, cam_SN in enumerate(list_camera_SN):
            dossier_camera = os.path.join(
                dossier_base,
                f"camera_{cam_SN}",
                f"{freq_acq}Hz",
                "images"
            )

            if not os.path.exists(dossier_camera):
                continue  # Passe à la caméra suivante si dossier introuvable

            # Filtre les fichiers .tiff de cette caméra
            fichiers = [
                f for f in os.listdir(dossier_camera)
                if f.startswith(f'Basler__{cam_SN}__') and f.endswith('.tiff')
            ]

            if not fichiers:
                continue

            fichiers_tries = natsorted(fichiers)
            dernier_fichier = fichiers_tries[-1]
            derniers_chiffres = extraire_derniers_chiffres(dernier_fichier)

            # Mise à jour uniquement si un nouveau fichier est détecté
            if derniers_chiffres != derniers_chiffres_precedent[idx_cam]:
                derniers_chiffres_precedent[idx_cam] = derniers_chiffres
                chemin_image = os.path.join(dossier_camera, dernier_fichier)

                try:
                    img = imread(chemin_image)
                    axes[idx_cam].clear()
                    axes[idx_cam].imshow(img, cmap='gray' if img.ndim == 2 else None)
                    axes[idx_cam].axis('off')
                    axes[idx_cam].set_title(f"Cam {cam_SN}: {dernier_fichier}", fontsize=8)
                except Exception as e:
                    print(f"Erreur image {chemin_image}: {e}")

        # =====================================================
        # 2. LECTURE DE LA DERNIÈRE LIGNE DU FICHIER PHIDGET
        # =====================================================
        try:
            with open(name_phidget_file, 'r') as file:
                # Lit la dernière ligne du fichier (optimisé)
                file.seek(0, os.SEEK_END)
                position = file.tell()
                ligne = ""
                while position >= 0:
                    file.seek(position)
                    char = file.read(1)
                    if char == "\n":
                        if ligne.strip():  # Ignore les lignes vides
                            break
                        ligne = ""
                    else:
                        ligne = char + ligne
                    position -= 1

                # Traite la ligne
                ligne = ligne.strip()
                if ligne and not ligne.startswith("time") and not ligne.startswith("new data"):
                    valeurs = ligne.split()
                    if len(valeurs) >= 5:  # Format attendu: time vr0 vr1 vr2 vr3
                        temps, vr0, vr1, vr2, vr3 = valeurs[:5]
                        texte_phidget.set_text(
                            f"t = {temps} s | VR0 = {vr0} | VR1 = {vr1} | VR2 = {vr2} | VR3 = {vr3}"
                        )
        except FileNotFoundError:
            texte_phidget.set_text("⚠️ Fichier Phidget introuvable !")
        except Exception as e:
            texte_phidget.set_text(f"⚠️ Erreur Phidget: {str(e)[:50]}")  # Affiche les 50 premiers caractères

        # =====================================================
        # 3. RAFRAÎCHISSEMENT DE LA FIGURE
        # =====================================================
        fig.canvas.draw()
        fig.canvas.flush_events()
        time.sleep(intervalle_affichage)
############################################



# --- Lancer l'acquisition caméra ---
acquisition_thread = threading.Thread(
    target=make_acquisition,
    args=(freq_acq, cam_array, list_camera_SN, dossier_base, True, 1000),
    daemon=True
)
acquisition_thread.start()


# --- Lancer l'acquisition Phidgets ---
phidget_thread = threading.Thread(
    target=interface_phidget_bridge_forbraziliantest,
    daemon=True
)
phidget_thread.start()


# Attendre que les premières images soient disponibles
time.sleep(2)


# --- Affichage caméra ---
try:
    fig, axes = plt.subplots(len(list_camera_SN))
    afficher_images_et_dataphidget()

except KeyboardInterrupt:
    print("\nArrêt du programme...")

finally:
    print("Arrêt de l'acquisition...")

26


In [7]:
%matplotlib qt